# ModernBERT KNN & Ensemble Trainer (Refactored)

This notebook is a refactored version of the training script. It supports both KNN-based training and Ensemble training for hate speech detection. It uses a structured, object-oriented approach.

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_recall_fscore_support, confusion_matrix, balanced_accuracy_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    EvalPrediction,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from transformers import pipeline
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import Any, Dict, List, Union, Optional
import warnings
import dataHandler as dh # Ensure dataHandler.py is in the same directory

# CheckList Imports
try:
    from checklist.pred_wrapper import PredictorWrapper
    from checklist.test_types import MFT
except ImportError:
    print("CheckList library not found. Please install it using `pip install checklist`.")

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

## 1. Setup and Configuration

In [ ]:
def setup_device():
    """Checks compute device availability and sets up the device."""
    print("="*60)
    print("CHECKING COMPUTE DEVICE STATUS")
    print("="*60)
    
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"✓ CUDA available: True")
        print(f"✓ CUDA device count: {torch.cuda.device_count()}")
        print(f"✓ Current device: {torch.cuda.current_device()}")
        print(f"✓ Device name: {torch.cuda.get_device_name(0)}")
        try:
            torch.cuda.empty_cache()
            print(f"✓ CUDA cache cleared successfully")
        except Exception as e:
            print(f"⚠ Warning clearing CUDA cache: {e}")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
         device = torch.device("mps")
         print("✓ MPS (Apple Silicon) available: True")
    else:
        device = torch.device("cpu")
        print("✓ CUDA not available, using CPU")
        
    return device

device = setup_device()
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
@dataclass
class Config:
    """Configuration for training and evaluation."""
    # Paths
    knn_datasets: Dict[str, str] = None
    output_dir_base: str = "models_knn_tox_refactored"
    
    # Model
    model_id: str = "tomh/toxigen_roberta"
    num_labels: int = 2
    labels: List[str] = None
    
    # Ensemble Settings
    use_ensemble: bool = False
    identity_terms: List[str] = field(default_factory=lambda: [
        'asian', 'black', 'chinese', 'jewish', 'latino', 'lgbtq', 
        'mental_dis', 'mexican', 'middle_east', 'muslim', 
        'native_american', 'physical_dis', 'women'
    ])
    
    # Data Processing
    max_length: int = 140
    test_size: float = 0.2
    random_state: int = 42
    outlier_lower_q: float = 0.0005
    outlier_upper_q: float = 0.9995
    
    # Training
    batch_size: int = 32
    learning_rate: float = 2e-5
    num_epochs: int = 4
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    logging_steps: int = 50
    eval_steps: int = 4000
    save_steps: int = 4000
    save_limit: int = 3
    
    def __post_init__(self):
        if self.knn_datasets is None:
            self.knn_datasets = {
                'k5': 'selected_knn_new/samples_knn_k5_percent100.csv',
                'k100': 'selected_knn_new/samples_knn_k100_percent100.csv',
                'k1000': 'selected_knn_new/samples_knn_k1000_percent100.csv',
            }
        if self.labels is None:
            self.labels = ["no hate", "hate"]
            
    @property
    def label2id(self):
        return {l: str(i) for i, l in enumerate(self.labels)}
        
    @property
    def id2label(self):
        return {str(i): l for i, l in enumerate(self.labels)}
        
    @property
    def output_dir(self):
        """Dynamic output directory based on model ID and mode."""
        model_name_clean = self.model_id.replace("/", "_")
        mode = "ensemble" if self.use_ensemble else "knn"
        return f"{self.output_dir_base}/{model_name_clean}/{mode}"

# Initialize Config
config = Config()
# config.use_ensemble = True # Uncomment to enable ensemble mode
print(f"Configuration initialized. Output directory: {config.output_dir}")

## 2. Data Processing

Encapsulates data loading, cleaning, density weight calculation, and tokenization.

In [ ]:
class DataProcessor:
    """Handles data loading, preprocessing, and tokenization."""
    
    def __init__(self, config: Config, tokenizer):
        self.config = config
        self.tokenizer = tokenizer

    def load_and_preprocess(self, file_path: str) -> DatasetDict:
        """Loads CSV, cleans data, calculates density weights, and splits into train/test."""
        print(f"Loading dataset from {file_path}")
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")
            
        df = pd.read_csv(file_path)
        
        # Basic cleaning
        if 'density' not in df.columns:
            raise ValueError(f"'density' column missing in {file_path}")
            
        # Ensure labels are binary integers
        if df['label'].dtype == 'object':
             df['label'] = df['label'].apply(lambda x: 1 if x == 'hate' else 0)
        
        if df['density'].isna().any():
            print(f"Filling {df['density'].isna().sum()} missing density values with median.")
            df['density'] = df['density'].fillna(df['density'].median())
            
        # Remove outliers
        orig_len = len(df)
        lower = df['density'].quantile(self.config.outlier_lower_q)
        upper = df['density'].quantile(self.config.outlier_upper_q)
        df = df[(df['density'] >= lower) & (df['density'] <= upper)]
        print(f"Removed {orig_len - len(df)} outliers based on density.")
        
        # Calculate density weights (normalized to [0.1, 2.0])
        min_d, max_d = df['density'].min(), df['density'].max()
        if min_d == max_d:
            print("Warning: All density values are the same. Setting weights to 1.0")
            df['density_weight'] = 1.0
        else:
            df['density_weight'] = 0.1 + (df['density'] - min_d) / (max_d - min_d) * 1.9
            
        # Split
        df_train = df.sample(frac=1-self.config.test_size, random_state=self.config.random_state)
        df_test = df.drop(df_train.index)
        
        # Convert to HF Dataset
        cols = ['text', 'label', 'density_weight']
        # Ensure columns exist
        cols = [c for c in cols if c in df.columns]
        
        train_ds = Dataset.from_pandas(df_train[cols])
        test_ds = Dataset.from_pandas(df_test[cols])
        
        # Cleanup index columns
        for ds in [train_ds, test_ds]:
            for col in ds.column_names:
                if col.startswith('__index'):
                    ds = ds.remove_columns(col)
                    
        return DatasetDict({"train": train_ds, "test": test_ds})

    def tokenize(self, batch):
        """Tokenization function to be used with dataset.map"""
        tokenized = self.tokenizer(
            batch['text'], 
            padding='max_length', 
            truncation=True, 
            max_length=self.config.max_length
        )
        # Add density weight if present
        if 'density_weight' in batch:
            tokenized['density_weight'] = batch['density_weight']
        
        # Rename label -> labels for Trainer
        if 'label' in batch:
            tokenized['labels'] = batch['label']
        elif 'labels' in batch:
            tokenized['labels'] = batch['labels']
            
        return tokenized

    def prepare_knn_datasets(self) -> Dict[str, DatasetDict]:
        """Process all KNN datasets defined in config."""
        processed = {}
        for name, path in self.config.knn_datasets.items():
            print(f"\nProcessing {name}...")
            try:
                ds_dict = self.load_and_preprocess(path)
                tokenized_ds = ds_dict.map(self.tokenize, batched=True)
                processed[name] = tokenized_ds
                print(f"Successfully processed {name}")
            except Exception as e:
                print(f"Error processing {name}: {e}")
        return processed
        
    def prepare_ensemble_datasets(self) -> Dict[str, DatasetDict]:
        """Process datasets for each identity term using dataHandler."""
        processed = {}
        print(f"Preparing ensemble datasets for terms: {self.config.identity_terms}")
        
        for term in self.config.identity_terms:
            print(f"\nProcessing term: {term}...")
            try:
                # Use dataHandler to get dataset
                dataset = dh.toxigenDataset(term, test_size=self.config.test_size)
                
                # Rename label column if needed
                if "label" in dataset["train"].features.keys():
                    dataset = dataset.rename_column("label", "labels")
                
                # Tokenize
                tokenized_ds = dataset.map(self.tokenize, batched=True)
                processed[term] = tokenized_ds
                print(f"Successfully processed {term}")
                
            except Exception as e:
                print(f"Error processing {term}: {e}")
                
        return processed

@dataclass
class DataCollatorWithDensityWeights(DataCollatorWithPadding):
    """Custom collator to preserve density_weight during batch creation."""
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Extract density weights if they exist
        density_weights = None
        if 'density_weight' in features[0]:
            density_weights = [f.pop('density_weight', 1.0) for f in features]
        
        # Remove 'text' field if it exists (Trainer expects only tensors)
        for f in features:
            f.pop('text', None) 
            
        batch = super().__call__(features)
        
        # Add weights back as tensor if they existed
        if density_weights is not None:
            batch['density_weight'] = torch.tensor(density_weights, dtype=torch.float32)
            
        return batch

## 3. Custom Trainer and Metrics

Defines the custom trainer that uses density weights for loss calculation.

In [ ]:
class DensityWeightedTrainer(Trainer):
    """Trainer that uses density weights in loss calculation."""
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        density_weights = inputs.pop("density_weight", None)
        labels = inputs.pop("labels", None)
        
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        if labels is not None:
            loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
            loss_per_sample = loss_fct(logits, labels)
            
            if density_weights is not None:
                density_weights = density_weights.to(loss_per_sample.device)
                loss = (loss_per_sample * density_weights).mean()
            else:
                loss = loss_per_sample.mean()
                
            outputs.loss = loss
            return (loss, outputs) if return_outputs else loss
            
        return (outputs.loss, outputs) if return_outputs else outputs.loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """Save density weights during evaluation for metrics computation."""
        if not hasattr(self, "_saved_density_weights"):
            self._saved_density_weights = []
            
        if "density_weight" in inputs:
            dw = inputs["density_weight"]
            if torch.is_tensor(dw):
                dw = dw.detach().cpu().numpy()
            self._saved_density_weights.append(dw)
            
        return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys)

    def evaluation_loop(self, dataloader, description, prediction_loss_only=None, ignore_keys=None, metric_key_prefix="eval"):
        output = super().evaluation_loop(dataloader, description, prediction_loss_only, ignore_keys, metric_key_prefix)
        
        # If we have saved weights, re-compute metrics with them
        if self.compute_metrics is not None and hasattr(self, "_saved_density_weights") and self._saved_density_weights:
            merged_dw = np.concatenate(self._saved_density_weights, axis=0)
            
            # Ensure lengths match (sometimes drop_last or padding might affect things, but usually fine in EvalLoop)
            if len(merged_dw) == len(output.label_ids):
                output.metrics.update(
                    self.compute_metrics(EvalPrediction(
                        predictions=output.predictions,
                        label_ids=output.label_ids,
                        inputs={"density_weight": merged_dw}
                    ))
                )
            else:
                print(f"Warning: Density weights length ({len(merged_dw)}) mismatch with labels ({len(output.label_ids)})")
                
            self._saved_density_weights = [] # Clear after use
            
        return output

def compute_metrics(eval_pred):
    """Compute weighted AUC and other metrics."""
    preds = eval_pred.predictions
    labels = eval_pred.label_ids
    inputs = getattr(eval_pred, "inputs", None)
    
    # Convert logits to probabilities
    if preds.ndim > 1:
        probs = np.exp(preds) / np.exp(preds).sum(-1, keepdims=True)
        probs = probs[:, 1]
    else:
        probs = 1 / (1 + np.exp(-preds))
        
    weights = None
    if inputs and "density_weight" in inputs:
        weights = inputs["density_weight"]
        
    try:
        auc = roc_auc_score(labels, probs, sample_weight=weights)
        f1 = f1_score(labels, np.argmax(preds, axis=1) if preds.ndim > 1 else (preds > 0.5).astype(int), average="weighted")
    except Exception as e:
        print(f"Error computing metrics: {e}")
        auc = 0.5
        f1 = 0.0
        
    return {"weighted_auc": auc, "f1": f1}

## 4. Training Manager

Encapsulates the training loop for both KNN and Ensemble modes.

In [ ]:
class ModelTrainer:
    """Manages model creation and training."""
    
    def __init__(self, config: Config, tokenizer):
        self.config = config
        self.tokenizer = tokenizer
        
    def create_model(self):
        return AutoModelForSequenceClassification.from_pretrained(
            self.config.model_id,
            num_labels=self.config.num_labels,
            label2id=self.config.label2id,
            id2label=self.config.id2label
        )
        
    def train_knn(self, dataset_name: str, tokenized_dataset: DatasetDict):
        """Train a single model on a KNN dataset."""
        print(f"\n{'='*40}\nTraining KNN Model for {dataset_name}\n{'='*40}")
        
        output_dir = f"{self.config.output_dir}/{dataset_name}"
        return self._run_training(tokenized_dataset, output_dir)
        
    def train_ensemble(self, term: str, tokenized_dataset: DatasetDict):
        """Train a single model for an ensemble term."""
        print(f"\n{'='*40}\nTraining Ensemble Model for {term}\n{'='*40}")
        
        output_dir = f"{self.config.output_dir}/{term}"
        return self._run_training(tokenized_dataset, output_dir)

    def _run_training(self, dataset, output_dir):
        # Clear cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        model = self.create_model().to(device)
        
        args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            learning_rate=self.config.learning_rate,
            num_train_epochs=self.config.num_epochs,
            warmup_ratio=self.config.warmup_ratio,
            weight_decay=self.config.weight_decay,
            bf16=torch.cuda.is_available(),
            fp16=not torch.cuda.is_available() and hasattr(torch, 'has_mps'),
            logging_strategy="steps",
            logging_steps=self.config.logging_steps,
            eval_strategy="steps",
            eval_steps=self.config.eval_steps,
            save_strategy="steps",
            save_steps=self.config.save_steps,
            save_total_limit=self.config.save_limit,
            load_best_model_at_end=True,
            metric_for_best_model="weighted_auc",
            greater_is_better=True,
            report_to="none", 
            include_for_metrics=["inputs"] # Important for custom trainer
        )
        
        trainer = DensityWeightedTrainer(
            model=model,
            args=args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["test"],
            compute_metrics=compute_metrics,
            data_collator=DataCollatorWithDensityWeights(self.tokenizer),
            callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
        )
        
        trainer.train()
        
        # Save final model
        final_path = f"{output_dir}/final"
        model.save_pretrained(final_path)
        self.tokenizer.save_pretrained(f"{output_dir}/tokenizer")
        print(f"Saved model to {final_path}")
        
        return final_path

## 5. Evaluation

Classes and functions for evaluating trained models, including MFT (CheckList) and Confusion Matrices.

In [ ]:
class Evaluator:
    """Handles model evaluation."""
    
    def __init__(self, config: Config):
        self.config = config
        
    def evaluate_direct(self, model_path, tokenizer_path, test_dataset):
        """Direct evaluation using manual batch processing."""
        print(f"Evaluating {model_path}...")
        
        try:
            tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
            model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
            model.eval()
            
            preds = []
            probs = []
            true_labels = test_dataset['label'].tolist() if hasattr(test_dataset['label'], 'tolist') else test_dataset['label']
            texts = test_dataset['text'].tolist() if hasattr(test_dataset['text'], 'tolist') else test_dataset['text']
            
            # Batch processing
            batch_size = 32
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i+batch_size]
                inputs = tokenizer(
                    batch_texts, 
                    padding=True, 
                    truncation=True, 
                    return_tensors="pt", 
                    max_length=self.config.max_length
                ).to(device)
                
                with torch.no_grad():
                    outputs = model(**inputs)
                    logits = outputs.logits
                    batch_probs = torch.nn.functional.softmax(logits, dim=1)
                    
                preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                probs.extend(batch_probs[:, 1].cpu().numpy())
                
            # Metrics
            acc = accuracy_score(true_labels, preds)
            bal_acc = balanced_accuracy_score(true_labels, preds)
            prec, rec, _, _ = precision_recall_fscore_support(true_labels, preds, average='binary')
            try:
                auc = roc_auc_score(true_labels, probs)
            except:
                auc = 0.5
                
            results = {
                "Accuracy": acc,
                "Balanced Accuracy": bal_acc,
                "Precision": prec,
                "Recall": rec,
                "AUC": auc
            }
            
            print(f"Results: {results}")
            
            # Plot Confusion Matrix
            self.plot_confusion_matrix(true_labels, preds, title=f"Confusion Matrix: {model_path.split('/')[-2]}")
            
            return results
            
        except Exception as e:
            print(f"Error evaluating {model_path}: {e}")
            return None

    def evaluate_mft(self, model_path, tokenizer_path, test_data_path):
        """Run CheckList MFT evaluation."""
        print(f"\nRunning MFT Evaluation for {model_path}...")
        try:
            # Load model and tokenizer wrapped for CheckList
            model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
            tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
            model.eval()
            
            def predict_proba(inputs):
                # CheckList passes a list of strings
                tokenized = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt", max_length=self.config.max_length).to(device)
                with torch.no_grad():
                    outputs = model(**tokenized)
                    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                return probs.cpu().numpy()

            wrapped_pp = PredictorWrapper.wrap_softmax(predict_proba)
            
            # Load test data for MFT (assuming CSV format compatible with MFT)
            # Note: This requires a specific MFT setup. Here we demonstrate a basic setup.
            # In a real scenario, you might load a specific MFT suite.
            # For now, we'll skip complex MFT suite loading and just show where it would go.
            print("MFT evaluation placeholder - requires defined MFT test suite.")
            # Example:
            # test = MFT.from_persistence(test_data_path)
            # test.run(wrapped_pp)
            # test.summary()
            
        except Exception as e:
            print(f"Error in MFT evaluation: {e}")

    def plot_confusion_matrix(self, y_true, y_pred, title="Confusion Matrix"):
        """Plots a confusion matrix."""
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(title)
        plt.show()

    def plot_comparison(self, results: Dict[str, Dict[str, float]]):
        """Plots a bar chart comparing models."""
        if not results:
            print("No results to plot.")
            return
            
        df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"})
        df_melted = df.melt(id_vars="Model", var_name="Metric", value_name="Score")
        
        plt.figure(figsize=(12, 6))
        sns.barplot(data=df_melted, x="Model", y="Score", hue="Metric")
        plt.title("Model Comparison")
        plt.ylim(0, 1.0)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

## 6. Main Execution

Orchestrates the entire workflow based on configuration.

In [ ]:
# 1. Initialize Config & Tokenizer
config = Config()
# Toggle this to switch between KNN and Ensemble modes
# config.use_ensemble = True 

tokenizer = AutoTokenizer.from_pretrained(config.model_id)
processor = DataProcessor(config, tokenizer)
trainer_manager = ModelTrainer(config, tokenizer)
evaluator = Evaluator(config)

trained_models = {}

if config.use_ensemble:
    print("STARTING ENSEMBLE TRAINING PIPELINE")
    # Prepare Data
    processed_datasets = processor.prepare_ensemble_datasets()
    
    if not processed_datasets:
        print("No datasets processed for ensemble.")
    else:
        # Train Models
        for term, ds in processed_datasets.items():
            path = trainer_manager.train_ensemble(term, ds)
            trained_models[term] = path
            
else:
    print("STARTING KNN TRAINING PIPELINE")
    # Prepare Data
    processed_datasets = processor.prepare_knn_datasets()
    
    if not processed_datasets:
        print("No datasets processed for KNN.")
    else:
        # Train Models
        for name, ds in processed_datasets.items():
            path = trainer_manager.train_knn(name, ds)
            trained_models[name] = path

print(f"\nTraining completed. Models saved in {config.output_dir}")

In [ ]:
# 4. Evaluate
# Uncomment to run evaluation
# results = {}
# for name, path in trained_models.items():
#     # Use the test split for evaluation
#     test_ds = processed_datasets[name]["test"].to_pandas()
#     
#     # Direct Evaluation (Metrics + Confusion Matrix)
#     metrics = evaluator.evaluate_direct(path, f"{path}/../tokenizer", test_ds)
#     if metrics:
#         results[name] = metrics
#     
#     # MFT Evaluation (Placeholder call)
#     # evaluator.evaluate_mft(path, f"{path}/../tokenizer", "path/to/mft/suite")
# 
# # Compare Models
# evaluator.plot_comparison(results)